# Rise of the Microagent
Many small LFM3 models, each in its own Hermes profile, on one Mac. Fake data only.

**Theory:** a monolith agent does everything; microagents each do one job with a contract (JSON in, JSON out), like microservices.

## 0. Servers
Start them with `bash mac/run_mac.sh serve`. Each microagent has its own model and port: router 8081 (d1-2B), scribe 8082 (LFM3-Audio-2B), receipts 8083 (LFM3-VL-2B), desk 8084 (LFM3-1.7B).

In [ ]:
import json, subprocess, urllib.request, os
SB = os.path.expanduser('~/proai-tmp/lfm3/sandbox')
for p in (8081, 8082, 8083, 8084):
    print(p, urllib.request.urlopen(f'http://127.0.0.1:{p}/health').read())

## 1. Router: a decision model, not a chat model
d1-2B answers several typed questions in one forward pass, each with a probability. That is the dispatcher in the microservices picture.

In [ ]:
import sys; sys.path.insert(0, 'agents/router'); import service as router
print(json.dumps(router.ask('Subject: URGENT verify your password now. Click here.'), indent=2))

In [ ]:
router.main(SB)

## 2. Scribe: audio in, text out
Voice memos are the most personal data you have. They never leave the laptop.

In [ ]:
subprocess.run(['python3', 'agents/scribe/service.py', SB], check=True)
print(open(f'{SB}/out/transcripts/memo1.txt').read())

## 3. Receipts: vision model with a JSON contract

In [ ]:
subprocess.run(['python3', 'agents/receipts/service.py', SB], check=True)
print(open(f'{SB}/out/expenses.csv').read())

## 4. Desk: LFM3-1.7B inside Hermes
`AIAgent(base_url=..., enabled_toolsets=['file'])`: the Hermes tool loop, pointed at a local endpoint, with only the file tools.

In [ ]:
subprocess.run(['python3', 'agents/desk/run_desk.py', SB], check=True)

## 5. Grade it
Objective checks only: label keys, an exact calendar slot, fuzzy to-do matching, exact receipt totals, and a draft that is saved, not sent.

In [ ]:
subprocess.run(['python3', 'grade/grade.py', SB], env={**os.environ, 'ANSWERS': os.path.expanduser('~/proai-tmp/lfm3/answers/monday')})

## 6. When would you fine-tune one of these?
When the format is fixed, the volume is high, prompting still fails the grader, and you have a few hundred or more labelled examples. See Liquid's cookbook notebooks (`lfm3_sft_with_trl`, `sft_for_decision_models_with_trl`) and the wind-turbine LFM3-VL example.